# eCommerce EDA — Normalized (Parquet)

Runs on the **normalized** star schema produced by `04_normalize.py`:

| table | rows | grain |
|-------|------|-------|
| `categories` | 691 | category_id → category_code |
| `products` | 206,876 | product_id → category_id, brand |
| `events` (fact) | ~110M | one row per event (view/cart/purchase) |

Parquet is columnar + compressed (14GB CSV → 3.4GB), so scans are faster than the raw CSV and brand/category come from small dimension tables via join.

**Heavy aggregation → DuckDB; charts → pandas/matplotlib.**

## Setup — load the star schema

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Resolve data/normalized relative to repo root (works from root or eda/).
ROOT = Path.cwd()
for cand in (ROOT, ROOT.parent):
    if (cand / "data" / "normalized").is_dir():
        ROOT = cand
        break
NORM = ROOT / "data" / "normalized"

need = ["events.parquet", "products.parquet", "categories.parquet"]
missing = [f for f in need if not (NORM / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {NORM}. Run `python eda/04_normalize.py` first."
    )

con = duckdb.connect()  # in-memory
# Load as TABLES (not views) so repeated queries hit RAM, not disk.
for name in ("events", "products", "categories"):
    con.sql(
        f"CREATE TABLE {name} AS "
        f"SELECT * FROM read_parquet('{(NORM / (name + '.parquet')).as_posix()}')"
    )

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

print("Loaded:")
display(q("""
    SELECT 'events' AS tbl, count(*) AS rows FROM events
    UNION ALL SELECT 'products', count(*) FROM products
    UNION ALL SELECT 'categories', count(*) FROM categories
"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded:


,tbl,rows
0,events,109950743
1,products,206876
2,categories,691


## 1. Overview

In [2]:
q("""
    SELECT
        count(*)                     AS total_events,
        min(event_time)              AS start_time,
        max(event_time)              AS end_time,
        count(DISTINCT user_id)      AS unique_users,
        count(DISTINCT product_id)   AS unique_products,
        count(DISTINCT user_session) AS unique_sessions
    FROM events
""").T

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0
total_events,109950743
start_time,2019-10-01 00:00:00
end_time,2019-11-30 23:59:59
unique_users,5316649
unique_products,206876
unique_sessions,23016650


## 2. Conversion funnel

In [ ]:
funnel = q("""
    SELECT
        count(*) FILTER (WHERE event_type='view')     AS views,
        count(*) FILTER (WHERE event_type='cart')     AS carts,
        count(*) FILTER (WHERE event_type='purchase') AS purchases
    FROM events
""").iloc[0]

print(f"view → cart     : {100*funnel.carts/funnel.views:5.2f}%")
print(f"cart → purchase : {100*funnel.purchases/funnel.carts:5.2f}%")
print(f"view → purchase : {100*funnel.purchases/funnel.views:5.2f}%")

stages = ["view", "cart", "purchase"]
vals = [funnel.views, funnel.carts, funnel.purchases]
ax = plt.subplot()
bars = ax.barh(stages[::-1], vals[::-1], color=["#54A24B", "#F58518", "#4C78A8"])
ax.set_title("Conversion funnel")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
for b, v in zip(bars, vals[::-1]):
    ax.text(v, b.get_y()+b.get_height()/2, f" {v/1e6:.1f}M", va="center")
plt.tight_layout(); plt.show()

## 3. Revenue by brand  *(join events × products)*

Brand now lives in the `products` dimension — this is where normalization pays off.

In [ ]:
brand = q("""
    SELECT p.brand,
           count(*)           AS purchases,
           round(sum(e.price)) AS revenue
    FROM events e
    JOIN products p USING (product_id)
    WHERE e.event_type='purchase' AND p.brand IS NOT NULL
    GROUP BY p.brand ORDER BY revenue DESC LIMIT 15
""")
display(brand)

ax = brand[::-1].plot.barh(x="brand", y="revenue", legend=False, color="#72B7B2")
ax.set_title("Top 15 brands by revenue")
ax.set_xlabel("revenue")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
plt.tight_layout(); plt.show()

## 4. Revenue by category  *(join events × products × categories)*

In [ ]:
cat = q("""
    SELECT c.category_code,
           count(*)            AS purchases,
           round(sum(e.price))  AS revenue,
           round(avg(e.price),2) AS avg_price
    FROM events e
    JOIN products p   USING (product_id)
    JOIN categories c USING (category_id)
    WHERE e.event_type='purchase' AND c.category_code IS NOT NULL
    GROUP BY c.category_code ORDER BY revenue DESC LIMIT 15
""")
display(cat)

ax = cat[::-1].plot.barh(x="category_code", y="revenue", legend=False, color="#4C78A8")
ax.set_title("Top 15 categories by revenue")
ax.set_xlabel("revenue")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
plt.tight_layout(); plt.show()

## 5. Category-level funnel  *(where do categories lose buyers?)*

Per top-level category (first segment of `category_code`), the view→purchase rate.

In [ ]:
catfunnel = q("""
    WITH j AS (
        SELECT split_part(c.category_code, '.', 1) AS top_cat, e.event_type
        FROM events e
        JOIN products p   USING (product_id)
        JOIN categories c USING (category_id)
        WHERE c.category_code IS NOT NULL
    )
    SELECT top_cat,
           count(*) FILTER (WHERE event_type='view')     AS views,
           count(*) FILTER (WHERE event_type='purchase') AS purchases,
           round(100.0*count(*) FILTER (WHERE event_type='purchase')
                     / NULLIF(count(*) FILTER (WHERE event_type='view'),0), 2) AS conv_pct
    FROM j GROUP BY top_cat
    HAVING views > 100000
    ORDER BY conv_pct DESC
""")
display(catfunnel)

ax = catfunnel[::-1].plot.barh(x="top_cat", y="conv_pct", legend=False, color="#54A24B")
ax.set_title("View → purchase conversion by top category")
ax.set_xlabel("conversion %")
plt.tight_layout(); plt.show()

## 6. Daily revenue trend

In [ ]:
daily = q("""
    SELECT date_trunc('day', event_time)::DATE AS day,
           round(sum(price)) AS revenue
    FROM events WHERE event_type='purchase'
    GROUP BY day ORDER BY day
""")

ax = daily.plot(x="day", y="revenue", legend=False, color="#4C78A8")
ax.set_title("Daily revenue (Oct–Nov 2019)")
ax.set_ylabel("revenue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
peak = daily.loc[daily["revenue"].idxmax()]
ax.annotate(f"{peak.day}", xy=(peak.day, peak.revenue),
            xytext=(0, 10), textcoords="offset points", ha="center")
plt.tight_layout(); plt.show()

## 7. Day-of-week × hour purchase heatmap

In [ ]:
hm = q("""
    SELECT extract(dow  FROM event_time) AS dow,
           extract(hour FROM event_time) AS hour,
           count(*) AS purchases
    FROM events WHERE event_type='purchase'
    GROUP BY dow, hour
""")
pivot = hm.pivot(index="dow", columns="hour", values="purchases").fillna(0)
days = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

fig, ax = plt.subplots(figsize=(11, 3.5))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(24)); ax.set_xticklabels(range(24))
ax.set_yticks(range(7));  ax.set_yticklabels(days)
ax.set_xlabel("hour (UTC)"); ax.set_title("Purchases: day-of-week × hour")
fig.colorbar(im, ax=ax, label="purchases")
plt.tight_layout(); plt.show()

## 8. Purchase price distribution

In [ ]:
hist = q("""
    SELECT floor(least(price, 2000)/50)*50 AS price_bin, count(*) AS n
    FROM events WHERE event_type='purchase'
    GROUP BY price_bin ORDER BY price_bin
""")
stats = q("""
    SELECT round(avg(price),2) AS mean,
           round(median(price),2) AS median,
           round(quantile(price,0.9),2) AS p90
    FROM events WHERE event_type='purchase'
""").iloc[0]
print(dict(stats))

ax = plt.subplot()
ax.bar(hist["price_bin"], hist["n"], width=45, color="#4C78A8")
ax.axvline(stats["median"], color="#E45756", ls="--", label=f"median ${stats['median']:.0f}")
ax.set_title("Purchase price distribution (capped at $2000)")
ax.set_xlabel("price ($)"); ax.set_ylabel("purchases"); ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
plt.tight_layout(); plt.show()

## 9. Top repeat buyers

In [ ]:
q("""
    SELECT user_id, count(*) AS purchases, round(sum(price)) AS spent
    FROM events WHERE event_type='purchase'
    GROUP BY user_id ORDER BY purchases DESC LIMIT 10
""")